# 9) Rolling horizon — Exercises

**Goals:** lay out the windows of a rolling horizon and evaluate it fairly, write your own rolling horizon and match PyPSA's exactly, measure how much look-ahead is worth, and find out how precisely a terminal water value has to be known.

The setup cell holds the lecture's week, its window model `window(hours, e_start, terminal_value)`, `fuel_cost`, the perfect-foresight benchmark and `pypsa_rolling` for checking. Power in MW, energy in MWh, money in € per week.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- the lecture's week and window model (same seed, same numbers) ----
T = 168
h = np.arange(T)
hour, day = h % 24, h // 24
rng = np.random.default_rng(3)
LOAD = (2000 + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0) + 750 * np.exp(-((hour - 19) / 2.0) ** 2)
        - 150 * (day >= 5) + rng.normal(0, 30, T))
cloud = np.repeat([0.9, 0.6, 1.0, 0.8, 0.4, 1.0, 0.9], 24)
FREE = (2500 * 0.75 * np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * cloud
        + 1000 * np.clip(0.3 + 0.25 * np.sin(2 * np.pi * h / 96), 0.02, 0.9) + 300.0)
THERMAL = pd.DataFrame({"p_nom": [350, 900, 600, 800, 400], "mc": [12.0, 89.1, 94.2, 104.1, 128.9]},
                       index=pd.Index(["nuclear", "ccgt", "coal", "imports", "ocgt"], name="gen"))
INFLOW, E_MAX, TURBINE, E_START, WATER_VALUE = 120.0, 30_000.0, 1000.0, 3000.0, 94.2
PERFECT = 9_828_300.2                       # lecture: perfect foresight, ending at the starting level


def window(hours, e_start, terminal_value=0.0):
    """The lecture's window LP. Returns generation (DataFrame), turbine (MW) and level (MWh) per hour."""
    n_t = len(hours)
    m = linopy.Model()
    t, G = pd.RangeIndex(n_t, name="t"), THERMAL.index
    g = m.add_variables(lower=0, upper=xr.DataArray(np.tile(THERMAL.p_nom.to_numpy(), (n_t, 1)), coords=[t, G]), name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(FREE[hours], coords=[t]), name="free")
    d = m.add_variables(lower=0, upper=TURBINE, coords=[t], name="turbine")
    s = m.add_variables(lower=0, coords=[t], name="spill")
    e = m.add_variables(lower=0, upper=E_MAX, coords=[t], name="level")
    m.add_constraints(e - e.shift(t=1) + d + s == INFLOW, name="water", mask=xr.DataArray(np.arange(n_t) > 0, coords=[t]))
    m.add_constraints(e.sel(t=0) + d.sel(t=0) + s.sel(t=0) == INFLOW + e_start, name="water_first_hour")
    m.add_constraints(g.sum("gen") + f + d == xr.DataArray(LOAD[hours], coords=[t]), name="balance")
    m.add_objective((g * xr.DataArray(THERMAL.mc.to_numpy(), coords=[G])).sum() - terminal_value * e.sel(t=n_t - 1))
    m.solve(solver_name="highs", output_flag=False)
    return g.solution.to_pandas(), d.solution.to_numpy(), e.solution.to_numpy()


def fuel_cost(gen):
    return float((gen * THERMAL.mc.to_numpy()).sum().sum())


def pypsa_rolling(horizon, overlap):
    """PyPSA's rolling horizon on a fresh network. Returns (fuel cost, level at the end of the week)."""
    n = pypsa.Network()
    n.set_snapshots(h)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=LOAD)
    n.add("Generator", "free", bus="HR", p_nom=FREE.max(), p_max_pu=FREE / FREE.max(), marginal_cost=0)
    for gen, r in THERMAL.iterrows():
        n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc)
    n.add("StorageUnit", "reservoir", bus="HR", p_nom=TURBINE, max_hours=E_MAX / TURBINE, p_min_pu=0.0,
          inflow=INFLOW, state_of_charge_initial=E_START, cyclic_state_of_charge=False)
    n.optimize.optimize_with_rolling_horizon(horizon=horizon, overlap=overlap, solver_name="highs",
                                             log_to_console=False, progress=False)
    return (float((n.generators_t.p[THERMAL.index] * THERMAL.mc).sum().sum()),
            float(n.storage_units_t.state_of_charge["reservoir"].iloc[-1]))

### Warm-ups

**1. Laying out the windows**

Write `windows(T, commit, lookahead)` returning a list of (planned hours, committed hours) pairs, both as `range`s, for a rolling horizon over T hours. The last windows are cut off at T. Check: committing 24 h with 24 h of look-ahead over a week gives 7 windows, and every hour is committed exactly once.

In [2]:
def windows(T: int, commit: int, lookahead: int) -> list:
    """(planned hours, committed hours) for each window of a rolling horizon."""
    ### BEGIN SOLUTION
    out = []
    for start in range(0, T, commit):
        out.append((range(start, min(T, start + commit + lookahead)), range(start, min(T, start + commit))))
    return out
    ### END SOLUTION

In [3]:
w = windows(168, 24, 24)
assert len(w) == 7 and w[0] == (range(0, 48), range(0, 24)) and w[-1] == (range(144, 168), range(144, 168))
committed = [hour for _, c in w for hour in c]
assert committed == list(range(168)), "every hour committed exactly once"
assert len(windows(168, 48, 0)) == 4 and windows(10, 4, 3)[-1] == (range(8, 10), range(8, 10))

**2. A fair comparison**

Write `adjusted_cost(fuel_cost, end_level)`: the fuel cost plus the water used beyond the starting level, valued at the water value (water saved beyond the start counts as a credit). Then explain in one sentence why two schedules with different fuel costs can both be optimal.

In [4]:
def adjusted_cost(fuel_cost: float, end_level: float) -> float:
    """Fuel cost (EUR) corrected for the change in stored water, valued at WATER_VALUE."""
    ### BEGIN SOLUTION
    return fuel_cost + WATER_VALUE * (E_START - end_level)
    ### END SOLUTION

In [5]:
assert np.isclose(adjusted_cost(9_575_977.0, 0.0), 9_858_577.0, atol=1), "lecture: day by day, no water value"
assert np.isclose(adjusted_cost(9_926_406.0, 4041.0), 9_828_300.0, atol=1), "lecture: own run with water value"
assert np.isclose(adjusted_cost(10_109_768.0, 5988.0), 9_828_300.0, atol=1), "lecture: PyPSA run with water value"

<!-- BEGIN SOLUTION -->
A schedule that leaves more water in the reservoir has spent less of its free fuel, so its lower fuel bill is not a saving: valuing the water left behind at what it is worth to the future puts both on the same footing — and when the water value equals the cost of the plant the water would replace (here coal at 94.2 €/MWh), storing a MWh and using it are worth exactly the same, so schedules that split the water differently have the same adjusted cost and are equally optimal.
<!-- END SOLUTION -->

### Core

**3. Your own rolling horizon**

Write `rolling(commit, lookahead, terminal_value=0.0)` using `windows` and the setup's `window`: solve each window from the level the previous one handed over, keep only its committed hours. Return the fuel cost, the level at the end of the week and the level after every hour. It must reproduce PyPSA's rolling horizon **exactly** in the myopic case.

In [6]:
def rolling(commit: int, lookahead: int, terminal_value: float = 0.0) -> dict:
    """Fuel cost (EUR), final level (MWh) and hourly levels of a rolling horizon over the week."""
    ### BEGIN SOLUTION
    e, gens, levels = E_START, [], []
    for planned, committed in windows(T, commit, lookahead):
        g, _, lev = window(np.array(planned), e, terminal_value)
        k = len(committed)
        gens.append(g.iloc[:k])
        levels.append(lev[:k])
        e = lev[k - 1]
    levels = np.concatenate(levels)
    return {"fuel_cost": fuel_cost(pd.concat(gens, ignore_index=True)), "end_level": levels[-1], "levels": levels}
    ### END SOLUTION

In [7]:
mine = rolling(24, 0)
py_cost, py_end = pypsa_rolling(24, 0)
assert np.isclose(mine["fuel_cost"], py_cost, rtol=1e-9) and np.isclose(mine["end_level"], py_end, atol=1e-6)
assert np.allclose(mine["levels"][23::24], 0.0, atol=1e-6), "the myopic reservoir is empty at every midnight"
valued = rolling(24, 0, WATER_VALUE)
assert np.isclose(adjusted_cost(valued["fuel_cost"], valued["end_level"]), PERFECT, atol=1.0), "the right water value closes the gap"
print(f"own = PyPSA, day by day: {mine['fuel_cost']:,.0f} EUR")

own = PyPSA, day by day: 9,575,977 EUR


**4. How much is look-ahead worth?**

Commit 24 hours and sweep the look-ahead over [0, 12, 24, 48, 72, 96, 144] hours. Report the adjusted cost and the gap to perfect foresight. Is more look-ahead always better?

In [8]:
def lookahead_sweep(values) -> pd.DataFrame:
    """Gap to perfect foresight (EUR, adjusted) for each look-ahead length (hours)."""
    ### BEGIN SOLUTION
    rows = {}
    for la in values:
        r = rolling(24, la)
        adj = adjusted_cost(r["fuel_cost"], r["end_level"])
        rows[la] = {"adjusted_cost": adj, "gap": adj - PERFECT, "end_level": r["end_level"]}
    return pd.DataFrame(rows).T.rename_axis("lookahead_h")
    ### END SOLUTION

In [9]:
sweep = lookahead_sweep([0, 12, 24, 48, 72, 96, 144])
assert (sweep.gap >= -1e-6).all(), "no rolling horizon beats perfect foresight"
assert sweep.loc[144, "gap"] < sweep.loc[0, "gap"]
sweep.round(0)

             adjusted_cost      gap  end_level
lookahead_h                                   
0                9858577.0  30276.0        0.0
12               9856523.0  28223.0        0.0
24               9851362.0  23062.0        0.0
48               9841409.0  13109.0        0.0
72               9833702.0   5402.0        0.0
96               9833702.0   5402.0        0.0
144              9833702.0   5402.0        0.0

**5. Interpretation**

What does the sweep say about how far ahead an operator must look in this week — and why does even the longest look-ahead not reach perfect foresight? (Hint: look at the end levels, and at what the last windows of the week can see.)

<!-- BEGIN SOLUTION -->
The gap shrinks as the windows see more of the week — from 30,276 € with no look-ahead to 5,402 € with 72 hours — and then stops: 96 and 144 hours give exactly the same 5,402 €. Three days of look-ahead already show each window the next evening peaks it should save water for; looking further adds nothing new in this week, whose pattern repeats day after day.

Even the longest look-ahead does not reach perfect foresight, because every window still ends empty: water left at the end of a window is worth nothing to it, and the last windows of the week cannot see beyond Sunday at all. So the rolling horizon finishes the week with 0 MWh, while perfect foresight hands back the 3,000 MWh it started with. The water that should have been kept is used in the last hours instead, in hours where it displaces cheaper plants than it is worth. The 5,402 € is consistent with about 1,060 MWh replacing the CCGT (89.1 €/MWh) instead of coal (94.2 €/MWh), at 5.1 €/MWh each. A longer window cannot fix this; only a value for the water at the end of the window can.
<!-- END SOLUTION -->

### Challenge

**6. How precisely must the water value be known?**

Run the day-by-day horizon with terminal water values of [0, 60, 89.1, 92, 94.2, 96, 104.1, 120, 200] €/MWh and report the adjusted cost gap and the end level for each. For which values is the gap zero? What happens above 94.2 €/MWh?

In [10]:
def water_value_sweep(values) -> pd.DataFrame:
    """Gap to perfect foresight (EUR) and end level (MWh) of the day-by-day horizon for each terminal water value."""
    ### BEGIN SOLUTION
    rows = {}
    for v in values:
        r = rolling(24, 0, v)
        adj = adjusted_cost(r["fuel_cost"], r["end_level"])
        rows[v] = {"gap": adj - PERFECT + 0.0, "end_level": r["end_level"], "fuel_cost": r["fuel_cost"]}
    return pd.DataFrame(rows).T.rename_axis("water_value")
    ### END SOLUTION

In [11]:
wv = water_value_sweep([0.0, 60.0, 89.1, 92.0, 94.2, 96.0, 104.1, 120.0, 200.0])
assert np.isclose(wv.loc[94.2, "gap"], 0.0, atol=1.0)
assert (wv.gap >= -1.0).all()
(wv.round(0) + 0.0)

                 gap  end_level   fuel_cost
water_value                                
0.0          30276.0        0.0   9575977.0
60.0         30276.0        0.0   9575977.0
89.1         26585.0        0.0   9572285.0
92.0             0.0     1059.0   9645475.0
94.2             0.0     4041.0   9926406.0
96.0             0.0    20547.0  11481189.0
104.1         4681.0    21019.0  11530415.0
120.0        25873.0    23160.0  11753245.0
200.0        25873.0    23160.0  11753245.0

**7. Interpretation**

Describe the pattern: what goes wrong when the water value is too low, and what when it is too high? Are the two kinds of error similar in size — and what does that suggest to an operator who is unsure of the water value?

<!-- BEGIN SOLUTION -->
Too low a water value (60 €/MWh or less, and even 89.1 €/MWh) changes nothing: the window still sees leftover water as worth less than the CCGT's fuel, empties the reservoir every day and misses by 26,585–30,276 €. Every value tried strictly between the CCGT's 89.1 and the imports' 104.1 €/MWh — 92, 94.2 and 96 — closes the gap to zero, even though they end the week with very different amounts of water (1,059 to 20,547 MWh): within that band the reservoir never spends water to replace the CCGT and never lets the imports or the OCGT run while it still holds water; whether a MWh replaces coal or is kept makes no difference, because the adjusted cost values the water left over at 94.2 €/MWh — exactly what it saves by replacing coal. Too high a value is wrong in the other direction: from 104.1 €/MWh the window would rather keep water than replace even the imports, hoards up to 23,160 MWh and lets imports and the OCGT run, costing up to 25,873 €.

So the water value does not need to be exact — it needs to lie in the right band between the marginal costs of the plants the water competes with. The errors on both sides are of similar size here, so an operator unsure of the value should aim for the middle of the plausible band, and should know which plants' costs bound it: those are the thresholds where the schedule changes. In a real system with many plants and a smooth supply curve the bands are narrower, and the long-term model that estimates water values matters more.
<!-- END SOLUTION -->